## Recomendación de Películas usando Sentence Transformers

Este proceso utiliza el modelo preentrenado `all-MiniLM-L6-v2` de **Sentence Transformers** para calcular las representaciones vectoriales de los géneros de las películas en el dataset. A continuación, se comparan estas representaciones utilizando la similitud del coseno para recomendar películas similares a una película de entrada.

### Flujo de trabajo:

1. **Carga de Datos**: Se carga el dataset de películas en formato Parquet, el cual contiene información como la sinopsis, géneros y puntuación de cada película.
2. **Preprocesamiento de Géneros**: Los géneros de cada película se procesan para ser usados como texto, dividiéndolos en listas de géneros separados por comas.
3. **Cálculo de Representaciones Vectoriales**:
   - Se utiliza el modelo `all-MiniLM-L6-v2` de **Sentence Transformers** para generar un vector de características a partir de la lista de géneros de cada película.
   - Cada película tiene asignado un vector basado en la concatenación de sus géneros.
4. **Recomendación de Películas**:
   - Dado el título de una película, se busca la película en el dataset y se obtiene su vector.
   - Se calcula la similitud del coseno entre el vector de la película de entrada y los vectores de todas las demás películas.
   - Se recomienda un número determinado de películas similares basado en la mayor similitud.

### Resultado:

El modelo devuelve las películas más similares a la película introducida, ordenadas según su puntuación de similitud, junto con los siguientes detalles:
- **sinopsis**: Una breve descripción de la película.
- **generos**: Los géneros de la película.
- **puntuacion**: La puntuación promedio de la película.

### Ejemplo de Resultados:

| sinopsis | generos | puntuacion |
|----------|---------|------------|
| Mientras trabajan en una avería subterránea, los operarios luchan contra una amenaza inesperada. | [] | 7.626 |
| En la Tierra Media, el Señor Oscuro Saurón crea un ejército para conquistar el mundo. | [] | 8.418 |
| Dom Cobb es un ladrón hábil, el mejor de todos, que roba secretos de la mente. | [] | 8.369 |
| Los soldados tailandeses inexpertos luchan con valentía en la jungla contra enemigos implacables. | [] | 3.900 |
| Shaun Boswell es un chico que no acaba de encajar hasta que encuentra el mundo de las carreras de coches. | [] | 6.500 |

### Advertencia:
Durante la carga del modelo, se mostró una advertencia relacionada con el sistema de caché de **Hugging Face** en Windows. Esto se debe a la falta de soporte de *symlinks* (enlaces simbólicos) en el sistema, lo cual no afectará el rendimiento en general, pero puede requerir más espacio en disco para almacenar los archivos del modelo.


In [10]:
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# Cargar el dataset
df = pd.read_parquet("C:/Users/jugas/OneDrive/Escritorio/Movie recommender/MVP_sistema_recomendacion/proyecto/data/movies_filtrado.parquet")

# Preprocesar los géneros
df["generos"] = df["generos"].apply(lambda x: x.split(",") if isinstance(x, str) else [])

# Cargar el modelo pre-entrenado de Sentence Transformers
model = SentenceTransformer('all-MiniLM-L6-v2')

# Función para obtener la representación vectorial de los géneros de una película
def get_movie_vector(genres, model):
    genres_text = " ".join(genres)  # Unir los géneros en una sola cadena
    genre_vector = model.encode([genres_text])[0]  # Obtener el vector de la representación de los géneros
    return genre_vector

# Calcular los vectores de todas las películas
df["vector"] = df["generos"].apply(lambda x: get_movie_vector(x, model))

# Función para recomendar películas similares
def recommend_movies(movie_title, df, model, top_n=5):
    movie_row = df[df["sinopsis"].str.contains(movie_title, case=False, na=False)]
    if movie_row.empty:
        return "Película no encontrada."
    
    movie_vector = movie_row.iloc[0]["vector"].reshape(1, -1)
    similarities = cosine_similarity(movie_vector, np.stack(df["vector"].values))
    df["similarity"] = similarities[0]
    return df.sort_values(by="similarity", ascending=False).head(top_n)[["sinopsis", "generos", "puntuacion"]]

# Ejemplo de recomendación
print(recommend_movies("Matrix", df, model))


                                            sinopsis generos  puntuacion
0  En la Tierra Media, el Señor Oscuro Saurón cre...      []       8.418
1  Dom Cobb es un ladrón hábil, el mejor de todos...      []       8.369
4  Cuando surge una nueva amenaza capaz de destru...      []       7.246
6  En una misión de rescate de rehenes, el capitá...      []       8.496
9  Dos agentes de élite son secretamente asignado...      []       7.747


In [12]:
# Obtener las recomendaciones para varias películas
movies_to_test = ["Matrix", "Origen", "Cars", "Harry Potter"]
for movie in movies_to_test:
    print(f"Recomendaciones para {movie}:")
    print(recommend_movies(movie, df, model))
    print("\n")


Recomendaciones para Matrix:
                                            sinopsis generos  puntuacion
0  En la Tierra Media, el Señor Oscuro Saurón cre...      []       8.418
1  Dom Cobb es un ladrón hábil, el mejor de todos...      []       8.369
4  Cuando surge una nueva amenaza capaz de destru...      []       7.246
6  En una misión de rescate de rehenes, el capitá...      []       8.496
9  Dos agentes de élite son secretamente asignado...      []       7.747


Recomendaciones para Origen:
                                            sinopsis generos  puntuacion
0  En la Tierra Media, el Señor Oscuro Saurón cre...      []       8.418
1  Dom Cobb es un ladrón hábil, el mejor de todos...      []       8.369
4  Cuando surge una nueva amenaza capaz de destru...      []       7.246
6  En una misión de rescate de rehenes, el capitá...      []       8.496
9  Dos agentes de élite son secretamente asignado...      []       7.747


Recomendaciones para Cars:
                                   